목적:
YouTube 영상 URL을 입력받아 STT 서버에 요청하고,
최종 clean transcript와 metadata를 outputs/{video_id}.json 으로 저장한다.

이 노트북은 STT 품질 개선이나 후처리를 고도화하지 않는다.
출력 JSON은 후속 단계의 입력으로 사용한다.

In [41]:
"""
Cell 2. import

- json: STT 서버 응답 파싱 및 결과 저장
- os: 환경변수, 출력 디렉토리 처리
- re: video_id 추출, 노이즈 제거
- shutil: curl 존재 확인
- subprocess: curl 요청
- tempfile: 임시 다운로드 디렉토리
- time: 소요 시간 로그
- datetime: JSON created_at 생성
- pathlib.Path: 경로 처리
- yt_dlp: YouTube audio download
"""

import json
import os
import re
import shutil
import subprocess
import tempfile
import time
from datetime import datetime, timedelta, timezone
from pathlib import Path

import yt_dlp

In [42]:
"""
Cell 3. 설정값

여기에 둘 것:
BASE_URL
MODEL
LANGUAGE
RESPONSE_FORMAT
TIMESTAMP_GRANULARITIES
OUTPUT_DIR
YOUTUBE_URL

구성 원칙:
- STT 서버 주소는 환경변수 우선
- 모델명은 qwen3-asr 기본값
- 언어는 ko 기본값
- response_format은 verbose_json 유지
- 출력 디렉토리는 .../notebooks/outputs
- youtube_url은 여기서만 바꾼다
"""

BASE_URL = os.environ.get("STT_BASE_URL", "http://localhost:8080").rstrip("/")
MODEL = os.environ.get("STT_MODEL", "qwen3-asr")
LANGUAGE = os.environ.get("DEFAULT_LANGUAGE", "ko")
RESPONSE_FORMAT = os.environ.get("STT_RESPONSE_FORMAT", "verbose_json")
TIMESTAMP_GRANULARITIES = [
    x.strip()
    for x in os.environ.get("STT_TIMESTAMP_GRANULARITIES", "segment").split(",")
    if x.strip()
]
OUTPUT_DIR = "./outputs"
YOUTUBE_URL = "https://www.youtube.com/watch?v=V6qrICA17Dg"

# https://www.youtube.com/watch?v=
#  /  / ZYbKUUrbatI

In [43]:
"""
Cell 4. 노이즈 패턴 정의

역할:
Qwen3-ASR 출력에 섞일 수 있는 language, Korean asr text 계열 문자열 제거
여기는 실험하지 말고 현재 동작하는 패턴 유지.
"""

NOISE_PATTERNS = [
    re.compile(r"\blanguage\s+[A-Za-z][A-Za-z_-]*\s*<\s*asr[\s_-]*text\s*>", re.IGNORECASE),
    re.compile(r"language\s*(?:Korean\s*asr\s*text|Koreanasrtext)", re.IGNORECASE),
    re.compile(r"Korean\s*asr\s*text", re.IGNORECASE),
    re.compile(r"Koreanasrtext", re.IGNORECASE),
    re.compile(
        r"(?:(?<=^)|(?<=[\s\]\)])|(?<=[\uac00-\ud7af0-9]))language(?=$|[\s\uac00-\ud7af0-9])",
        re.IGNORECASE,
    ),
]

In [44]:
"""
Cell 5. 유틸 함수

포함 함수:

ensure_command(name)
extract_video_id(youtube_url)

역할:

ensure_command:
- curl 없으면 바로 실패

extract_video_id:
- YouTube URL에서 video_id 추출
- outputs/{video_id}.json 파일명 생성에 사용
- youtu.be/VIDEO_ID, youtube.com/watch?v=VIDEO_ID 형식 지원
"""


def ensure_command(name: str):
    if not shutil.which(name):
        raise RuntimeError(f"Required command not found in PATH: {name}")


def extract_video_id(youtube_url):
    patterns = [
        r"youtu\.be/([A-Za-z0-9_-]+)",
        r"[?&]v=([A-Za-z0-9_-]+)",
    ]
    for pattern in patterns:
        match = re.search(pattern, youtube_url)
        if match:
            return match.group(1)
    return "unknown_video"

In [45]:
"""
Cell 6. STT 텍스트 정리 함수

포함 함수:
clean_asr_text(text)
build_clean_transcript(payload)

역할:
clean_asr_text:
- 노이즈 패턴 제거
- 중복 공백 정리
- 문장부호 앞 공백 제거

build_clean_transcript:
- verbose_json의 segments를 paragraph 단위 clean text로 병합
- segment 간 공백이 2.5초 이상이면 문단 분리
- segments가 없으면 payload["text"] 기준으로 fallback
"""


def clean_asr_text(text):
    cleaned = text
    for pattern in NOISE_PATTERNS:
        cleaned = pattern.sub("", cleaned)
    cleaned = re.sub(r"\s+", " ", cleaned)
    cleaned = re.sub(r"\s+([,.;:!?])", r"\1", cleaned)
    return cleaned.strip()


def build_clean_transcript(payload):
    segments = payload.get("segments")
    if not segments:
        return clean_asr_text(str(payload.get("text", "")))

    paragraphs = []
    current = []
    previous_end = None

    start_t = time.time()

    for segment in segments:
        text = clean_asr_text(str(segment.get("text", "")))
        if not text:
            continue

        start = segment.get("start")
        if current and previous_end is not None and start is not None and float(start) - previous_end >= 2.5:
            paragraphs.append(" ".join(current))
            current = []

        current.append(text)
        if segment.get("end") is not None:
            previous_end = float(segment["end"])

    if current:
        paragraphs.append(" ".join(current))

    elapsed = time.time() - start_t
    if elapsed > 3.0:
        print(f"[TIME] build_clean_transcript took {elapsed:.2f} seconds")
    return "\n\n".join(paragraphs)

In [46]:
"""
Cell 7. 오디오 다운로드 함수

포함 함수:
download_audio(youtube_url, target_dir)

역할:
- yt-dlp로 YouTube audio 다운로드
- tempfile 내부에 audio 파일 생성
- 다운로드 소요 시간 출력
- 다운로드 실패 시 RuntimeError
- yt-dlp metadata를 후속 JSON 저장 단계로 전달
"""


def download_audio(youtube_url, target_dir):
    ydl_opts = {
        "outtmpl": str(target_dir / "audio.%(ext)s"),
        "format": "bestaudio/best",
        "noplaylist": True,
        "quiet": True,
        "continuedl": False,
        "nooverwrites": True,
    }
    print("[INFO] Downloading full audio with yt-dlp...")
    start_t = time.time()
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(youtube_url, download=True)
        audio_path = Path(ydl.prepare_filename(info))

    elapsed = time.time() - start_t
    print(f"[TIME] Audio download took {elapsed:.2f} seconds")

    if not audio_path.is_file():
        raise RuntimeError(f"Downloaded file not found: {audio_path}")
    print(f"[INFO] downloaded_audio={audio_path}")
    return audio_path, info

In [47]:
"""
Cell 8. STT 서버 요청 함수

포함 함수:
request_transcription(audio_path)

역할:
- curl로 /v1/audio/transcriptions 호출
- file, model, language, response_format 전달
- timestamp_granularities[] 전달
- timeout은 7200 유지
- 실패 시 stderr/stdout 기반 RuntimeError
- 성공 시 JSON payload와 요청 소요 시간 반환
"""


def request_transcription(audio_path):
    ensure_command("curl")
    form_args = [
        "-F", f"file=@{audio_path}",
        "-F", f"model={MODEL}",
        "-F", f"language={LANGUAGE}",
        "-F", f"response_format={RESPONSE_FORMAT}",
    ]
    for granularity in TIMESTAMP_GRANULARITIES:
        form_args.extend(["-F", f"timestamp_granularities[]={granularity}"])

    print("[INFO] Requesting STT transcription from server...")
    start_t = time.time()
    proc = subprocess.run(
        [
            "curl",
            "-sS",
            "--fail-with-body",
            "-X", "POST",
            *form_args,
            f"{BASE_URL}/v1/audio/transcriptions",
        ],
        capture_output=True,
        text=True,
        check=False,
        timeout=7200,
    )

    elapsed = time.time() - start_t
    print(f"[TIME] STT transcription request took {elapsed:.2f} seconds")

    if proc.returncode != 0:
        message = proc.stderr.strip() or proc.stdout.strip()
        raise RuntimeError(f"STT request failed: {message}")

    payload = json.loads(proc.stdout or "{}")
    if not isinstance(payload, dict):
        raise RuntimeError(f"Unexpected transcription response type: {type(payload).__name__}")
    return payload, elapsed

In [48]:
"""
Cell 9. 출력 저장 함수

포함 함수:
save_stt_result(clean_text, payload, youtube_url, video_info, stt_elapsed_sec, output_dir)

역할:
- video_id 추출
- output_dir 생성
- clean transcript와 metadata를 outputs/{video_id}.json 으로 저장
- 저장 경로 반환

중요한 점:
clean transcript 내용은 transcript 필드에 그대로 저장
raw STT payload 전체는 저장하지 않음
segments는 stats.segment_count 계산에만 사용
"""


def save_stt_result(clean_text, payload, youtube_url, video_info, stt_elapsed_sec, output_dir):
    video_id = extract_video_id(youtube_url)
    os.makedirs(output_dir, exist_ok=True)

    upload_date = video_info.get("upload_date")
    if upload_date:
        upload_date = str(upload_date)
        if re.fullmatch(r"\d{8}", upload_date):
            upload_date = f"{upload_date[:4]}-{upload_date[4:6]}-{upload_date[6:8]}"

    created_at = datetime.now(timezone(timedelta(hours=9))).isoformat(timespec="seconds")
    result = {
        "created_at": created_at,
        "source": {
            "video_id": video_id,
            "url": youtube_url,
            "title": video_info.get("title"),
            "channel": video_info.get("uploader"),
            "upload_date": upload_date,
            "duration_sec": video_info.get("duration"),
            "language": video_info.get("language") or LANGUAGE,
        },
        "stt": {
            "model": MODEL,
            "elapsed_sec": round(float(stt_elapsed_sec), 2),
        },
        "stats": {
            "transcript_char_count": len(clean_text),
            "segment_count": len(payload.get("segments", [])),
        },
        "transcript": clean_text,
    }

    output_path = Path(output_dir) / f"{video_id}.json"
    with output_path.open("w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)

    return output_path

In [49]:
"""
Cell 10. 실행 셀

역할:
1. 임시 디렉토리 생성
2. YouTube audio 다운로드 및 metadata 확보
3. STT 서버 요청 및 소요 시간 확보
4. clean transcript 생성
5. outputs/{video_id}.json 저장
6. 최종 저장 경로 출력

이 셀이 실제 실행 진입점.

"""

output_path = None
clean_text = ""

with tempfile.TemporaryDirectory(prefix="yt-qwen-asr-") as tmp_dir:
    work_dir = Path(tmp_dir)
    try:
        audio_path, video_info = download_audio(YOUTUBE_URL, work_dir)
        result, stt_elapsed = request_transcription(audio_path)
        clean_text = build_clean_transcript(result)
        output_path = save_stt_result(
            clean_text,
            result,
            YOUTUBE_URL,
            video_info,
            stt_elapsed,
            OUTPUT_DIR,
        )
        print(f"[SUCCESS] STT result saved to {output_path}")
    except Exception as e:
        print(f"[FAIL] {type(e).__name__}: {e}")

[INFO] Downloading full audio with yt-dlp...


[TIME] Audio download took 4.85 seconds                    
[INFO] downloaded_audio=/tmp/yt-qwen-asr-gkdro43h/audio.webm
[INFO] Requesting STT transcription from server...
[TIME] STT transcription request took 27.51 seconds
[SUCCESS] STT result saved to outputs/V6qrICA17Dg.json


In [ ]:
"""
Cell 11. 반복 폭주 구간 감지 및 후처리
역할:
- STT 결과에서 같은 짧은 구절이 비정상적으로 반복되는 구간 감지
- 5회 이상 반복되는 phrase를 1회로 축약
- 원본 transcript는 보존하고 cleaned_transcript를 별도로 저장
- 어떤 반복이 제거됐는지 로그 출력
"""


def tokenize_for_repetition(text):
    return [
        {
            "text": match.group(0),
            "start": match.start(),
            "end": match.end(),
            "normalized": normalize_repetition_token(match.group(0)),
        }
        for match in re.finditer(r"\S+", text)
    ]


REPETITION_BOUNDARY_CHARS = " \t\n\r.,;:!?\"'“”‘’()[]{}<>…·"


def normalize_repetition_token(token):
    return token.strip(REPETITION_BOUNDARY_CHARS).lower()


def format_repetition_phrase(tokens):
    phrase = " ".join(token["text"] for token in tokens)
    phrase = re.sub(r"\s+", " ", phrase)
    return phrase.strip(REPETITION_BOUNDARY_CHARS)


def smallest_repeating_unit_size(phrase_tokens, min_ngram=2):
    for unit_size in range(min_ngram, len(phrase_tokens) + 1):
        if len(phrase_tokens) % unit_size != 0:
            continue
        unit = phrase_tokens[:unit_size]
        if all(
            phrase_tokens[start:start + unit_size] == unit
            for start in range(0, len(phrase_tokens), unit_size)
        ):
            return unit_size
    return len(phrase_tokens)


def find_repetition_loops(text, min_ngram=2, max_ngram=12, min_repeat=5):
    tokens = tokenize_for_repetition(text)
    loops = []
    i = 0

    while i < len(tokens):
        detected = None
        max_size = min(max_ngram, (len(tokens) - i) // min_repeat)

        for ngram_size in range(max_size, min_ngram - 1, -1):
            phrase_tokens = tuple(token["normalized"] for token in tokens[i:i + ngram_size])
            if not all(phrase_tokens):
                continue

            repeat_count = 1
            while True:
                next_start = i + repeat_count * ngram_size
                next_end = next_start + ngram_size
                if next_end > len(tokens):
                    break
                next_tokens = tuple(token["normalized"] for token in tokens[next_start:next_end])
                if next_tokens != phrase_tokens:
                    break
                repeat_count += 1

            if repeat_count >= min_repeat:
                unit_size = smallest_repeating_unit_size(phrase_tokens, min_ngram=min_ngram)
                total_repeat_count = repeat_count * ngram_size // unit_size
                original_start = tokens[i]["start"]
                original_end = tokens[i + total_repeat_count * unit_size - 1]["end"]
                kept_end = tokens[i + unit_size - 1]["end"]
                phrase = format_repetition_phrase(tokens[i:i + unit_size])
                detected = {
                    "start": original_start,
                    "end": original_end,
                    "replacement": text[original_start:kept_end],
                    "phrase": phrase,
                    "repeat_count": total_repeat_count,
                    "kept_count": 1,
                    "ngram_size": unit_size,
                }
                break

        if detected:
            loops.append(detected)
            i += detected["repeat_count"] * detected["ngram_size"]
        else:
            i += 1

    return loops


def clean_repetition_loops(text, keep_count=1):
    if keep_count != 1:
        raise ValueError("Only keep_count=1 is supported for now")

    loops = find_repetition_loops(text)
    if not loops:
        return text, []

    cleaned_parts = []
    previous_end = 0
    removed_repetitions = []

    for loop in loops:
        cleaned_parts.append(text[previous_end:loop["start"]])
        cleaned_parts.append(loop["replacement"])
        previous_end = loop["end"]
        removed_repetitions.append({
            "phrase": loop["phrase"],
            "repeat_count": loop["repeat_count"],
            "kept_count": loop["kept_count"],
        })

    cleaned_parts.append(text[previous_end:])
    cleaned = "".join(cleaned_parts)
    cleaned = re.sub(r"[ \t]{2,}", " ", cleaned)
    cleaned = re.sub(r"\n{3,}", "\n\n", cleaned)
    return cleaned.strip(), removed_repetitions


if output_path is None:
    output_path = Path(OUTPUT_DIR) / f"{extract_video_id(YOUTUBE_URL)}.json"

with Path(output_path).open("r", encoding="utf-8") as f:
    data = json.load(f)

original_transcript = data.get("transcript", "")
cleaned_transcript, removed_repetitions = clean_repetition_loops(original_transcript)
postprocess = data.setdefault("postprocess", {})

print("[Repetition Cleaning]\n")

if removed_repetitions:
    original_char_count = len(original_transcript)
    cleaned_char_count = len(cleaned_transcript)
    removed_char_count = original_char_count - cleaned_char_count
    removed_char_ratio = removed_char_count / original_char_count if original_char_count else 0.0

    data["cleaned_transcript"] = cleaned_transcript
    postprocess["repetition_cleaning"] = {
        "applied": True,
        "original_char_count": original_char_count,
        "cleaned_char_count": cleaned_char_count,
        "removed_char_count": removed_char_count,
        "removed_char_ratio": round(removed_char_ratio, 3),
        "removed_repetitions": removed_repetitions,
    }

    for repetition in removed_repetitions:
        print("Detected phrase:")
        print(f"\"{repetition['phrase']}\"\n")
        print(f"Repeat count: {repetition['repeat_count']}")
        print(f"Kept count: {repetition['kept_count']}\n")

    print(f"Original chars: {original_char_count}")
    print(f"Cleaned chars: {cleaned_char_count}")
    print(f"Removed chars: {removed_char_count}")
    print(f"Removed ratio: {removed_char_ratio:.1%}")
else:
    data.pop("cleaned_transcript", None)
    postprocess["repetition_cleaning"] = {
        "applied": False,
        "reason": "no_excessive_repetition_detected",
    }
    print("No excessive repetition detected.")

with Path(output_path).open("w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)


In [ ]:
"""
Cell 12. 결과 확인 셀

역할:

- 저장된 JSON 파일 열기
- 영상 metadata와 전체 글자 수 출력
- cleaned_transcript가 있으면 우선 사용해 앞부분 1500자 출력

목적:
STT가 정상적으로 되었는지 빠르게 육안 확인
전체 transcript를 노트북에 매번 다 출력할 필요는 없음. 너무 길어서 가독성만 떨어짐.
"""

if output_path is None:
    output_path = Path(OUTPUT_DIR) / f"{extract_video_id(YOUTUBE_URL)}.json"

with Path(output_path).open("r", encoding="utf-8") as f:
    data = json.load(f)

preview_length = 1500
source = data.get("source", {})
stats = data.get("stats", {})
transcript = data.get("transcript", "")
preview_transcript = data.get("cleaned_transcript", transcript)
preview_source = "cleaned_transcript" if "cleaned_transcript" in data else "transcript"

print(f"[INFO] File path: {output_path}")
print(f"[INFO] Video title: {source.get('title')}")
print(f"[INFO] Channel: {source.get('channel')}")
print(f"[INFO] Duration: {source.get('duration_sec')}")
print(f"[INFO] Total characters: {stats.get('transcript_char_count', len(transcript)):,}")
print(f"[INFO] Preview source: {preview_source}")
print("\n======= Transcript Preview =======\n")
print(preview_transcript[:preview_length])
if len(preview_transcript) > preview_length:
    print("\n... [truncated]")

[INFO] File path: outputs/V6qrICA17Dg.json
[INFO] Video title: 40 High Income Businesses Ideas to Start in 2026
[INFO] Channel: Business Money 
[INFO] Duration: 1965
[INFO] Total characters: 24,298

======= Transcript Preview =======

Welcome to Business Money. Success begins with the right idea, and 2026 brings powerful new opportunities. Here are the high-income businesses worth starting if you're ready to build lasting wealth and independence. The Kasori food dehydrator is a low-cost way to enter the dried food market. It has six stainless steel trays, a 600-watt power output, alanguage English<asr_text>A thirty-five degrees Celsius to seventy-three degrees Celsius temperature range and a forty-eight hour timer for hands-off operation. The glass door lets you check progress without opening it. Priced at one hundred fifty-nine dollars and ninety-nine cents, it's ideal for making jerky or fruit chips. Producing two kilograms of jerky daily at a nine per hundred gram selling price brin